In [1]:
import os
import json
from openai import OpenAI
from google.colab import userdata
from dotenv import load_dotenv
from openai import OpenAI

==============================================
## Initializing LLM Client
==============================================

In [2]:
load_dotenv()

try:
    client = OpenAI(api_key=userdata.get('OPENAI_APIKEY'))
except Exception as e:
    print(f"Error initializing ApenAI LLM client: {e} Please ensure that your environment variables are setup with a valid API key.")

==============================================
## Tool 1 - Domain Lookup Tool
==============================================

In [3]:
def lookup_domain_info_v2(domain: str) -> str:
    """Enrich company data and log raw JSON output."""
    mock_data = {
        "acmecorp.com": {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"},
        "widgetco.net": {"industry": "Manufacturing", "size": "100-250 employees", "revenue": "$10M - $25M"},
        "globalfin.org": {"industry": "Financial Services", "size": "5000+ employees", "revenue": "$1B+"},
    }
    info = mock_data.get(domain, {"industry": "Unknown", "size": "N/A", "revenue": "N/A"})
    json_output = json.dumps(info)
    print(f"[TOOL OUTPUT - lookup_domain_info]: {json_output}")
    return json_output

==============================================
## Tool 2 - Check CRM Hstory Tool
==============================================

In [4]:
def check_crm_history_v2(email: str) -> str:
    """Check CRM history and log raw JSON output."""
    mock_data = {
        "jane@acmecorp.com": {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."},
        "bob@widgetco.net": {"last_contact": "2025-12-01", "status": "Active Opportunity", "notes": "Discussed Q1 budget and product integration."},
        "default": {"last_contact": "N/A", "status": "No Record", "notes": "New lead, first contact opportunity."},
    }
    history = mock_data.get(email, mock_data["default"])
    json_output = json.dumps(history)
    print(f"[TOOL OUTPUT - check_crm_history]: {json_output}")
    return json_output

==============================================
## Tool 3 - Lead Score Calculator Tool
==============================================

In [5]:
def calculate_lead_score_v2(data_summary: str) -> str:
    """Calculate lead score and log raw JSON output."""
    data = json.loads(data_summary)
    score = "Low"
    if data["domain_info"].get("revenue", "").startswith("$1B+"):
        score = "High"
    elif data["crm_history"].get("status") == "Active Opportunity":
        score = "High"
    elif data["domain_info"].get("revenue", "").startswith("$50M"):
        score = "Medium"

    result = {"lead_score": score}
    json_output = json.dumps(result)
    print(f"[TOOL OUTPUT - calculate_lead_score]: {json_output}")
    return json_output

==============================================
## Function-mapping
==============================================

In [6]:
AVAILABLE_FUNCTIONS_V2 = {
    "lookup_domain_info_v2": lookup_domain_info_v2,
    "check_crm_history_v2": check_crm_history_v2,
    "calculate_lead_score_v2": calculate_lead_score_v2,
}

==============================================
## Defining Tool Schemas
==============================================

In [7]:
tools_schema = [
    {
        "type": "function",
        "function": {
            "name": "lookup_domain_info_v2",
            "description": "Retrieves general business information (industry, size, revenue) about a company based on its domain name.",
            "parameters": {
                "type": "object",
                "properties": {
                    "domain": {"type": "string", "description": "The company's domain name, e.g., 'acmecorp.com'"},
                },
                "required": ["domain"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "check_crm_history_v2",
            "description": "Checks the internal CRM system for past contact, status, and notes associated with a specific lead email.",
            "parameters": {
                "type": "object",
                "properties": {
                    "email": {"type": "string", "description": "The full email address of the lead."},
                },
                "required": ["email"],
            },
        },
    },
    {
        "type": "function",
        "function": {
            "name": "calculate_lead_score_v2",
            "description": "Calculates the priority score (High/Medium/Low) for a lead based on a summary of all collected domain and CRM history data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "data_summary": {"type": "string", "description": "A JSON string containing the combined domain_info and crm_history."},
                },
                "required": ["data_summary"],
            },
        },
    },
]

=====================================================================
## Initialize ReAct Agent Workflow
=====================================================================


In [8]:
def run_agent_with_detailed_tool_call(user_prompt: str):
    """Modified agent loop with raw JSON logging for tool calls and responses."""
    print(f"\n--- Running Lead Qualifier Agent with Detailed Logging ---")
    system_prompt = "You are an expert CRM Lead Qualifier Agent. Follow steps: 1. Identify domain 2. lookup_domain_info & check_crm_history 3. Combine JSON 4. calculate_lead_score 5. Summarize."
    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    collected_data = {}

    while True:
        print("\n[AI Thinking...]")
        response = client.chat.completions.create(model="gpt-4o", messages=messages, tools=tools_schema, tool_choice="auto")
        response_message = response.choices[0].message
        messages.append(response_message)

        if response_message.tool_calls:
            for tool_call in response_message.tool_calls:
                # LOG: Raw Tool Call from LLM
                print(f"[LLM TOOL CALL]: {json.dumps(tool_call.model_dump(), indent=2)}")

                function_name = tool_call.function.name
                function_to_call = AVAILABLE_FUNCTIONS_V2.get(function_name)
                function_args = json.loads(tool_call.function.arguments)

                if function_name == "calculate_lead_score_v2":
                    function_args = {"data_summary": json.dumps(collected_data)}

                function_result = function_to_call(**function_args)

                if function_name == "lookup_domain_info_v2":
                    collected_data["domain_info"] = json.loads(function_result)
                elif function_name == "check_crm_history_v2":
                    collected_data["crm_history"] = json.loads(function_result)

                tool_message = {
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "name": function_name,
                    "content": function_result
                }
                # LOG: Raw Tool response being sent back to LLM
                print(f"[SENT TO LLM]: {json.dumps(tool_message, indent=2)}")
                messages.append(tool_message)
        else:
            print("\n--- FINAL AGENT SUMMARY ---")
            print(response_message.content)
            break

==============================================
## Execute Test Scenario
==============================================

In [9]:
# Execute test scenario with ReAct loop and detailed logging
run_agent_with_detailed_tool_call('Please qualify this lead: jane@acmecorp.com')


--- Running Lead Qualifier Agent with Detailed Logging ---

[AI Thinking...]
[LLM TOOL CALL]: {
  "id": "call_JrAgiQSHAvGbcaNlZKgqLdiW",
  "function": {
    "arguments": "{\"domain\": \"acmecorp.com\"}",
    "name": "lookup_domain_info_v2"
  },
  "type": "function"
}
[TOOL OUTPUT - lookup_domain_info]: {"industry": "Software/SaaS", "size": "501-1000 employees", "revenue": "$50M - $100M"}
[SENT TO LLM]: {
  "role": "tool",
  "tool_call_id": "call_JrAgiQSHAvGbcaNlZKgqLdiW",
  "name": "lookup_domain_info_v2",
  "content": "{\"industry\": \"Software/SaaS\", \"size\": \"501-1000 employees\", \"revenue\": \"$50M - $100M\"}"
}
[LLM TOOL CALL]: {
  "id": "call_eFzqdFcckaUn175mp45i4boY",
  "function": {
    "arguments": "{\"email\": \"jane@acmecorp.com\"}",
    "name": "check_crm_history_v2"
  },
  "type": "function"
}
[TOOL OUTPUT - check_crm_history]: {"last_contact": "2025-11-15", "status": "Cold Lead", "notes": "Attended webinar, no follow-up yet."}
[SENT TO LLM]: {
  "role": "tool",
  "to